# Text-only stratified: Qwen vs LLaVA (paired analysis)

**Design:** 30 matched `data_split_seed` values (same protocol as `data/processed/seeds_list.txt`).  
**Metrics:** test macro F1 and test accuracy (both on the held-out test split for that seed).

**Paths (defaults):**
- Qwen: `experiments/textonly_qwen_stratified_splits_robustness/metrics/experiments/seed_*_results.json`
- LLaVA: `experiments/textonly_llava_stratified_robustness/metrics/experiments/seed_*_results.json`

**Accuracy scale:** LLaVA JSON stores test accuracy as a **percent** (0–100); Qwen stores a **fraction** (0–1). This notebook normalizes both to **fraction** before comparison.

**Hypothesis (one-sided, pre-registered pair):** Qwen > LLaVA on each metric.  
**Multiple testing:** two metrics → **Bonferroni** at family $\alpha=0.05$ (reject if $p < 0.025$ per metric). Uncorrected one-sided *p* values are also printed.

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

_p = Path.cwd().resolve()
for _ in range(12):
    if (_p / "experiments").is_dir() and (_p / "data").is_dir():
        PROJECT_ROOT = _p
        break
    _p = _p.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

QWEN_DIR = PROJECT_ROOT / "experiments" / "textonly_qwen_stratified_splits_robustness" / "metrics" / "experiments"
LLAVA_DIR = PROJECT_ROOT / "experiments" / "textonly_llava_stratified_robustness" / "metrics" / "experiments"


def _norm_acc(x: float) -> float:
    """Map stored test accuracy to fraction in [0, 1]."""
    x = float(x)
    if x > 1.5:
        return x / 100.0
    return x


def load_experiment(dir_path: Path, source: str) -> pd.DataFrame:
    rows = []
    if not dir_path.is_dir():
        print(f"Missing directory: {dir_path}")
        return pd.DataFrame()
    for path in sorted(dir_path.glob("seed_*_results.json")):
        m = re.match(r"seed_(\d+)_results\.json$", path.name)
        if not m:
            continue
        with open(path) as f:
            d = json.load(f)
        seed = int(d.get("data_split_seed", d.get("seed_value", int(m.group(1)))))
        tm = d["test_metrics"]
        f1 = float(tm["test_macro_f1"])
        acc = _norm_acc(tm["test_accuracy"])
        rows.append({"seed": seed, "source": source, "test_macro_f1": f1, "test_accuracy": acc})
    return pd.DataFrame(rows)


df_q = load_experiment(QWEN_DIR, "qwen")
df_l = load_experiment(LLAVA_DIR, "llava")

if df_q.empty or df_l.empty:
    raise FileNotFoundError(
        "Need JSON results under both experiment dirs. "
        f"Qwen n={len(df_q)}, LLaVA n={len(df_l)}"
    )

wide = df_q.merge(df_l, on="seed", suffixes=("_qwen", "_llava"), how="inner")
wide = wide.sort_values("seed").reset_index(drop=True)
wide["delta_f1"] = wide["test_macro_f1_qwen"] - wide["test_macro_f1_llava"]
wide["delta_acc"] = wide["test_accuracy_qwen"] - wide["test_accuracy_llava"]

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Paired rows: {len(wide)} (Qwen dir n={len(df_q)}, LLaVA dir n={len(df_l)})")
if len(wide) < 30:
    print("Warning: fewer than 30 paired seeds — check missing JSONs.")

display_cols = [
    "seed",
    "test_macro_f1_qwen",
    "test_macro_f1_llava",
    "delta_f1",
    "test_accuracy_qwen",
    "test_accuracy_llava",
    "delta_acc",
]
print("\n=== Per-seed (accuracy = fraction) ===")
display(wide[display_cols])


def summarize(s: pd.Series) -> str:
    a = np.asarray(s, dtype=float)
    return f"mean={a.mean():.4f}  std={a.std(ddof=1):.4f}  min={a.min():.4f}  max={a.max():.4f}"


print("\n=== Aggregate ===")
print("Test macro F1  | Qwen:  ", summarize(wide["test_macro_f1_qwen"]))
print("Test macro F1  | LLaVA:", summarize(wide["test_macro_f1_llava"]))
print("Test accuracy  | Qwen:  ", summarize(wide["test_accuracy_qwen"]))
print("Test accuracy  | LLaVA:", summarize(wide["test_accuracy_llava"]))
print("Delta F1 (Q−L):", summarize(wide["delta_f1"]))
print("Delta acc (Q−L):", summarize(wide["delta_acc"]))

wins_f1 = int((wide["delta_f1"] > 0).sum())
ties_f1 = int((wide["delta_f1"] == 0).sum())
loss_f1 = int((wide["delta_f1"] < 0).sum())
wins_a = int((wide["delta_acc"] > 0).sum())
ties_a = int((wide["delta_acc"] == 0).sum())
loss_a = int((wide["delta_acc"] < 0).sum())
print(f"\nWins / ties / losses (Qwen vs LLaVA): F1 → {wins_f1} {ties_f1} {loss_f1}; acc → {wins_a} {ties_a} {loss_a}")

# --- Plots ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(wide))
axes[0].plot(x, wide["test_macro_f1_qwen"], "o-", label="Qwen", alpha=0.85)
axes[0].plot(x, wide["test_macro_f1_llava"], "s-", label="LLaVA", alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(wide["seed"], rotation=90, fontsize=7)
axes[0].set_ylabel("Test macro F1")
axes[0].legend()
axes[0].set_title("Per-seed test macro F1")

axes[1].scatter(wide["test_macro_f1_llava"], wide["test_macro_f1_qwen"], alpha=0.75)
lo = min(wide["test_macro_f1_llava"].min(), wide["test_macro_f1_qwen"].min())
hi = max(wide["test_macro_f1_llava"].max(), wide["test_macro_f1_qwen"].max())
axes[1].plot([lo, hi], [lo, hi], "k--", alpha=0.5, label="y = x")
axes[1].set_xlabel("LLaVA test macro F1")
axes[1].set_ylabel("Qwen test macro F1")
axes[1].set_title("Qwen vs LLaVA (F1)")
axes[1].legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, wide["test_accuracy_qwen"], "o-", label="Qwen", alpha=0.85)
ax.plot(x, wide["test_accuracy_llava"], "s-", label="LLaVA", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(wide["seed"], rotation=90, fontsize=7)
ax.set_ylabel("Test accuracy (fraction)")
ax.set_title("Per-seed test accuracy")
ax.legend()
plt.tight_layout()
plt.show()

# --- Paired tests: H1 Qwen > LLaVA ---
alpha_family = 0.05
m_tests = 2
alpha_bonf = alpha_family / m_tests


def paired_report(name, q, l):
    q = np.asarray(q, dtype=float)
    l = np.asarray(l, dtype=float)
    d = q - l
    sd = float(np.std(d, ddof=1)) if len(d) > 1 else 0.0
    cohen_dz = float(np.mean(d) / sd) if sd > 0 else float("nan")
    tt = stats.ttest_rel(q, l, alternative="greater")
    # Wilcoxon: scipy >= 1.8 supports alternative
    try:
        wil = stats.wilcoxon(q, l, alternative="greater", method="auto")
    except TypeError:
        wil = stats.wilcoxon(q, l, alternative="greater")
    return {
        "metric": name,
        "mean_qwen": float(np.mean(q)),
        "mean_llava": float(np.mean(l)),
        "mean_diff_Q_minus_L": float(np.mean(d)),
        "std_diff": sd,
        "paired_Cohen_dz": cohen_dz,
        "t_statistic": float(tt.statistic),
        "p_t_one_sided": float(tt.pvalue),
        "p_wilcoxon_one_sided": float(wil.pvalue),
        "reject_t_bonferroni": bool(tt.pvalue < alpha_bonf),
        "reject_wilcoxon_bonferroni": bool(wil.pvalue < alpha_bonf),
    }


r_f1 = paired_report("test_macro_f1", wide["test_macro_f1_qwen"], wide["test_macro_f1_llava"])
r_acc = paired_report("test_accuracy", wide["test_accuracy_qwen"], wide["test_accuracy_llava"])
rep = pd.DataFrame([r_f1, r_acc])

print("\n=== Paired tests (one-sided: Qwen > LLaVA) ===")
print(f"Family alpha = {alpha_family}, Bonferroni per-metric alpha = {alpha_bonf} (m = {m_tests} metrics)")
display(rep)

print("\nInterpretation:")
for _, row in rep.iterrows():
    name = row["metric"]
    print(
        f"  {name}: mean diff = {row['mean_diff_Q_minus_L']:.4f}, "
        f"t one-sided p = {row['p_t_one_sided']:.4g}, "
        f"Wilcoxon p = {row['p_wilcoxon_one_sided']:.4g} | "
        f"reject H0 (Bonferroni t) = {row['reject_t_bonferroni']}"
    )
